In [0]:
from pyspark.sql.functions import *

In [0]:
def generate_hash_key(*columns):
    return sha2(concat_ws("|", *columns), 256)

In [0]:
def generate_hashdiff(*columns):
    return sha2(concat_ws("|", *columns), 256)

In [0]:
def standardize_name(column_name):
    return upper(trim(col(column_name)))

In [0]:
def validate_email(column_name):
    return col(column_name).rlike(
        "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"
    )

In [0]:
customer_df = spark.table("default.clean_customer")

customer_df.show(5)

+-----------+----------+-------------------+------+-------+--------------------+-------------+--------+
|Customer_ID|      Name|              Email|  City|Country|           Load_Date|Record_Source|Batch_ID|
+-----------+----------+-------------------+------+-------+--------------------+-------------+--------+
|       C001|Customer_1|customer1@gmail.com|City_2|  India|2026-07-13 10:01:...| Customer_CSV|       1|
|       C002|Customer_2|customer2@gmail.com|City_3|  India|2026-07-13 10:01:...| Customer_CSV|       1|
|       C003|Customer_3|customer3@gmail.com|City_4|  India|2026-07-13 10:01:...| Customer_CSV|       1|
|       C004|Customer_4|customer4@gmail.com|City_5|  India|2026-07-13 10:01:...| Customer_CSV|       1|
|       C005|Customer_5|customer5@gmail.com|City_6|  India|2026-07-13 10:01:...| Customer_CSV|       1|
+-----------+----------+-------------------+------+-------+--------------------+-------------+--------+
only showing top 5 rows


In [0]:
customer_df = customer_df \
    .withColumn(
        "Customer_HK",
        generate_hash_key(col("Customer_ID"))
    ) \
    .withColumn(
        "HashDiff",
        generate_hashdiff(
            col("Name"),
            col("Email"),
            col("City")
        )
    ) \
    .withColumn(
        "Customer_Name",
        standardize_name("Name")
    ) \
    .withColumn(
        "Valid_Email",
        validate_email("Email")
    )

In [0]:
customer_df.select(
    "Customer_ID",
    "Customer_Name",
    "Customer_HK",
    "HashDiff",
    "Valid_Email"
).show(10, truncate=False)

+-----------+-------------+----------------------------------------------------------------+----------------------------------------------------------------+-----------+
|Customer_ID|Customer_Name|Customer_HK                                                     |HashDiff                                                        |Valid_Email|
+-----------+-------------+----------------------------------------------------------------+----------------------------------------------------------------+-----------+
|C001       |CUSTOMER_1   |a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|8d7ef09c398a8fc4f2214c4ca3b47d8b5ba929fe0b21145c0ddb8634570357cd|true       |
|C002       |CUSTOMER_2   |fdc98571fc93e8482954dcc8034bda27f571bd691cd86510a849dce99759933d|5682cc4efd8ffa1d19650189ea009de35fbc251a30732ce3f70d412f20d561b0|true       |
|C003       |CUSTOMER_3   |64604487c74ef2fb0aca58ed3e951aacc1f537baf96c023b75441012e4b92880|add7d8c2cae5b60736a253725ef6c4c6caf7094d03344f6929bbdc3a73